# 다중 작업 승인 예제

에이전트가 한 번에 여러 도구를 호출하면, 대기 중인 작업(`action_requests`)도 여러 개가 됩니다.
이때 `decisions` 배열의 순서는 반드시 `action_requests`의 순서와 **정확히 일치**해야 합니다.

이 노트북은 이메일 발송 + 회의 예약을 한 번에 요청해서, 두 건의 승인 대기를 만들고
각각 다른 결정(첫 번째는 approve, 두 번째는 edit)을 내려봅니다.

**모델**: 로컬 Ollama `llama3.1:8b` (API 키 불필요). Ollama 서버가 켜져 있어야 합니다.

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

## 1. 도구 정의

세 도구 모두 승인이 필요하도록 설정합니다 (아래에서는 이 중 2개를 한 번에 호출시킵니다).

In [2]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """이메일을 전송합니다."""
    return f"Email sent to {recipient} with subject '{subject}'"


@tool
def schedule_meeting(participants: list[str], time: str) -> str:
    """회의를 예약합니다."""
    return f"Meeting scheduled at {time} with {', '.join(participants)}"


@tool
def create_document(title: str, content: str) -> str:
    """새 문서를 생성합니다."""
    return f"Document '{title}' created"

## 2. 모든 도구에 승인이 필요한 에이전트 생성

In [3]:
model = init_chat_model("ollama:llama3.1:8b", temperature=0)
# Claude API를 쓰려면 위 줄 대신:
# model = init_chat_model("claude-sonnet-4-5")

agent = create_agent(
    model=model,
    tools=[send_email, schedule_meeting, create_document],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "schedule_meeting": True,
                "create_document": True,
            },
        ),
    ],
    checkpointer=InMemorySaver(),
)

print("에이전트 생성 완료")

에이전트 생성 완료


## 3. 한 번에 두 작업 요청

"이메일 보내고 + 회의 예약해줘"를 한 번에 요청합니다. 모델이 두 도구를 모두 호출하면
`action_requests`에 2건이 대기하게 됩니다.

In [4]:
config = {"configurable": {"thread_id": "thread_multi"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use tools to email john@example.com with subject 'Project Update' "
                    "about progress, AND schedule a meeting with the team tomorrow at 2pm. "
                    "Do both using the available tools."
                ),
            }
        ]
    },
    config=config,
)

if "__interrupt__" in result:
    action_requests = result["__interrupt__"][0].value["action_requests"]
    print(f"승인 대기 중인 작업: {len(action_requests)}건\n")
    for i, action in enumerate(action_requests):
        print(f"[{i}] {action['name']}({action['args']})")
else:
    print("interrupt가 발생하지 않았습니다 (모델이 도구를 하나만 호출했을 수 있음 — 위 프롬프트를 조금 더 구체적으로 바꿔보세요)")

승인 대기 중인 작업: 2건

[0] send_email({'recipient': 'john@example.com', 'subject': 'Project Update', 'body': 'Progress update for the project.'})
[1] schedule_meeting({'participants': ['John', 'Alice', 'Bob'], 'time': 'tomorrow at 2pm'})


## 4. 작업별로 다른 결정 내리기

`decisions` 리스트의 순서는 `action_requests` 순서와 정확히 일치해야 합니다.
여기서는 도구 이름을 기준으로 결정을 만들어서, 실제 순서가 어떻게 나오든 안전하게 매칭합니다.

- `send_email` → **approve** (그대로 승인)
- `schedule_meeting` → **edit** (시간을 3시로 변경해서 승인)
- 그 외 → **approve**

In [5]:
decisions = []
for action in action_requests:
    if action["name"] == "schedule_meeting":
        decisions.append(
            {
                "type": "edit",
                "edited_action": {
                    "name": "schedule_meeting",
                    "args": {
                        "participants": action["args"].get("participants", []),
                        "time": "tomorrow at 3pm",  # 2pm -> 3pm으로 수정
                    },
                },
            }
        )
    else:
        decisions.append({"type": "approve"})

print("결정 순서:", [d["type"] for d in decisions])

result = agent.invoke(Command(resume={"decisions": decisions}), config=config)

print("\n최종 응답:")
print(result["messages"][-1].content)

결정 순서: ['approve', 'edit']



최종 응답:
The tool call response indicates that the email was sent successfully but the meeting time was changed from 2pm to 3pm.
